# Detección de sesgo entre árbitros de análisis de minerales

- Artículo completo en la plataforma: https://fuzzyfrog.ai/es/ai-lab/proyectos/negocios/deteccion-sesgo-arbitros-mineria-backtesting-machine-learning/
- Este notebook reconstruye, con datos sintéticos, el mismo flujo del proyecto original: pruebas de hipótesis para distinguir azar de sesgo, comparación de modelos, y backtesting contra los métodos usados actualmente.
- **Nota:** el dataset usado es sintético, con árbitros, clientes y magnitudes ficticias, generado para fines demostrativos. No contiene datos reales de la empresa ni de los árbitros originales. Un árbitro ficticio fue construido intencionalmente con sesgo, para poder demostrar que el método sí lo detecta.

## Diagrama de arquitectura

`Transacciones con arbitraje → prueba de hipótesis de sesgo → ingeniería de variables → modelo por árbitro / por elemento → clasificación gana-pierde → backtesting vs. promedio histórico y azar → perfil de tendencia por árbitro`

Las pruebas de hipótesis ocurren antes que cualquier modelo, responden si vale la pena seguir investigando un árbitro.

## Carga de datos

Se carga el dataset sintético de transacciones con arbitraje.

In [ ]:
import pandas as pd

data = pd.read_csv("outputs/dataset_sintetico_arbitraje_minerales.csv", parse_dates=["Fecha"])
print(f"Numero de registros: {data.shape[0]}")
data.head(3)

## Explicación de datos

- `An_Vendedor`, `An_Comprador`, `An_Arbitro`: resultado de cada análisis independiente para el elemento correspondiente.
- `Diferencia_Arbitro_Vendedor`: diferencia entre el resultado del árbitro y el del vendedor, la variable central de este análisis.
- `Arbitro`, `Cliente`, `Elemento`: dimensiones sobre las que se busca detectar patrones de tendencia.

In [ ]:
data[["Arbitro", "Cliente", "Elemento"]].nunique()

## Análisis de datos: prueba de hipótesis de sesgo

Antes de construir cualquier modelo, se prueba si la diferencia promedio entre el árbitro y el vendedor es estadísticamente distinta de cero, tanto a nivel agregado por árbitro como por cliente. Una prueba t de una muestra contra una media de cero es la forma más directa de distinguir una tendencia sistemática de una variación explicable por azar.

In [ ]:
from scipy import stats

resultados_sesgo = {}
for arbitro in data["Arbitro"].unique():
    diferencias = data[data["Arbitro"] == arbitro]["Diferencia_Arbitro_Vendedor"]
    t_stat, p_value = stats.ttest_1samp(diferencias, popmean=0)
    resultados_sesgo[arbitro] = {
        "diferencia_promedio": diferencias.mean(),
        "t_stat": t_stat,
        "p_value": p_value,
        "sesgo_significativo": p_value < 0.05,
    }

pd.DataFrame(resultados_sesgo).T.sort_values("diferencia_promedio", ascending=False)

**Lectura de los resultados:** un árbitro con una diferencia promedio distinta de cero y un p-value por debajo de 0.05 muestra evidencia estadística de una tendencia sistemática, no solo variación aleatoria. En el dataset sintético, el árbitro construido intencionalmente con sesgo aparece de forma clara con esta prueba.

## Análisis de datos: sesgo por cliente

Se repite la misma prueba, ahora agrupando por cliente, para ver si la tendencia depende también de con quién se está negociando, no solo de qué árbitro interviene.

In [ ]:
resultados_sesgo_cliente = {}
for (arbitro, cliente), grupo in data.groupby(["Arbitro", "Cliente"]):
    if len(grupo) < 5:
        continue
    diferencias = grupo["Diferencia_Arbitro_Vendedor"]
    t_stat, p_value = stats.ttest_1samp(diferencias, popmean=0)
    resultados_sesgo_cliente[(arbitro, cliente)] = {
        "diferencia_promedio": diferencias.mean(),
        "p_value": p_value,
        "n": len(grupo),
    }

df_sesgo_cliente = pd.DataFrame(resultados_sesgo_cliente).T
df_sesgo_cliente[df_sesgo_cliente["p_value"] < 0.05].sort_values("diferencia_promedio", ascending=False).head(10)

## Modelado: preparación de variables

Se codifica la fecha de forma cíclica y las variables categóricas con one-hot, y se preparan los datos para entrenar un modelo por árbitro.

In [ ]:
import numpy as np

def codificar_fecha_ciclica(df, col):
    df = df.copy()
    dia_del_anio = df[col].dt.dayofyear
    df[col + "_sin"] = np.sin(2 * np.pi * dia_del_anio / 365)
    df[col + "_cos"] = np.cos(2 * np.pi * dia_del_anio / 365)
    return df.drop(columns=[col])

data_modelo = codificar_fecha_ciclica(data, "Fecha")
data_modelo = pd.get_dummies(data_modelo, columns=["Cliente", "Elemento"])
data_modelo.head(2)

## Modelado: comparación de algoritmos por árbitro

Se entrena un modelo distinto para cada árbitro, comparando varios algoritmos de regresión, para predecir el resultado del árbitro a partir de las demás variables.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error

modelos_candidatos = {
    "Arbol de Decision": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "AdaBoost": AdaBoostRegressor(random_state=42),
    "SVR": SVR(),
}

columnas_excluir = ["An_Arbitro", "Arbitro", "Diferencia_Arbitro_Vendedor"]
resultados_por_arbitro = {}

for arbitro in data_modelo["Arbitro"].unique():
    df_arbitro = data_modelo[data_modelo["Arbitro"] == arbitro]
    X = df_arbitro.drop(columns=columnas_excluir)
    y = df_arbitro["An_Arbitro"]

    if len(df_arbitro) < 20:
        continue

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    resultados_por_arbitro[arbitro] = {}
    for nombre_modelo, modelo in modelos_candidatos.items():
        modelo.fit(X_train, y_train)
        y_pred = modelo.predict(X_test)
        resultados_por_arbitro[arbitro][nombre_modelo] = {
            "MAPE": mean_absolute_percentage_error(y_test, y_pred),
            "MSE": mean_squared_error(y_test, y_pred),
        }

for arbitro, resultados in resultados_por_arbitro.items():
    print(f"--- {arbitro} ---")
    print(pd.DataFrame(resultados).T)
    print()

## Modelado: clasificación gana / pierde

Se agrega un modelo de clasificación binaria que predice directamente a quién favorece el resultado final del árbitro, comprador o vendedor, sin necesitar el valor exacto.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

data_clasificacion = data.copy()
data_clasificacion["Target_gano_comprador"] = (
    (data_clasificacion["An_Comprador"] - data_clasificacion["An_Arbitro"]).abs()
    < (data_clasificacion["An_Vendedor"] - data_clasificacion["An_Arbitro"]).abs()
).astype(int)

X_clas = data_clasificacion[["An_Vendedor", "An_Comprador"]]
y_clas = data_clasificacion["Target_gano_comprador"]

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_clas, y_clas, test_size=0.2, random_state=42)

clf = RandomForestClassifier(n_estimators=200, random_state=42)
clf.fit(X_train_c, y_train_c)
y_pred_c = clf.predict(X_test_c)

print("Accuracy:", accuracy_score(y_test_c, y_pred_c))
print("Precision:", precision_score(y_test_c, y_pred_c))
print("Recall:", recall_score(y_test_c, y_pred_c))
print("F1:", f1_score(y_test_c, y_pred_c))

## Evaluación: backtesting contra los métodos actuales

El error del mejor modelo, Random Forest, se compara contra dos métodos simples ya usados en la práctica: el promedio histórico de diferencia por árbitro, y una elección al azar entre el valor del comprador y el del vendedor.

In [ ]:
import random

errores_modelo = {}
errores_promedio_historico = {}
errores_azar = {}

for arbitro in data_modelo["Arbitro"].unique():
    df_arbitro = data_modelo[data_modelo["Arbitro"] == arbitro]
    if len(df_arbitro) < 20:
        continue

    X = df_arbitro.drop(columns=columnas_excluir)
    y = df_arbitro["An_Arbitro"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Modelo entrenado
    rf = RandomForestRegressor(n_estimators=200, random_state=42)
    rf.fit(X_train, y_train)
    pred_modelo = rf.predict(X_test)
    errores_modelo[arbitro] = mean_absolute_percentage_error(y_test, pred_modelo)

    # Baseline 1: promedio historico de diferencia (Arbitro = Vendedor + diferencia promedio historica)
    diferencia_promedio_historica = df_arbitro.loc[X_train.index, "Diferencia_Arbitro_Vendedor"].mean()
    an_vendedor_test = data.loc[X_test.index, "An_Vendedor"]
    pred_promedio = an_vendedor_test + diferencia_promedio_historica
    errores_promedio_historico[arbitro] = mean_absolute_percentage_error(y_test, pred_promedio)

    # Baseline 2: eleccion al azar entre comprador y vendedor
    an_comprador_test = data.loc[X_test.index, "An_Comprador"]
    random.seed(42)
    pred_azar = [
        random.choice([v, c]) for v, c in zip(an_vendedor_test, an_comprador_test)
    ]
    errores_azar[arbitro] = mean_absolute_percentage_error(y_test, pred_azar)

comparacion = pd.DataFrame({
    "Modelo (Random Forest)": errores_modelo,
    "Promedio historico": errores_promedio_historico,
    "Azar": errores_azar,
})
comparacion

## Hallazgos principales

- La prueba de hipótesis de media cero, aplicada sobre la diferencia entre árbitro y vendedor, detecta con evidencia estadística qué árbitros muestran una tendencia sistemática y cuáles no, evitando sacar conclusiones de diferencias que podrían deberse solo al azar.
- Repetir la prueba por cliente permite matizar el hallazgo, una tendencia puede ser más fuerte con ciertos clientes que con otros.
- El modelo de Random Forest, entrenado por árbitro, superó consistentemente a los dos métodos usados actualmente en la práctica, validado con backtesting.
- El diseño del problema, traducir "¿hay sesgo intencional?" en una prueba estadística contestable con los datos disponibles, fue la parte más determinante del proyecto, más que la elección final del algoritmo.